# LeetCode 3378 — Count Connected Components in LCM Graph bằng Python

- Link: https://leetcode.com/problems/count-connected-components-in-lcm-graph/
- Độ khó: **Hard**
- Chủ đề: **Disjoint Set Union (DSU)**, **sàng theo bội số** (tổng điều hòa `t/1 + t/2 + … ≈ t ln t`), nhận xét "`lcm(a, b) ≤ t` ⇔ có bội chung `≤ t`"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countComponents(self, nums: List[int], threshold: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra số `> threshold` luôn **cô lập** (`lcm ≥` chính nó).
2. Chuyển điều kiện `lcm(a, b) ≤ t` thành: **tồn tại `m ≤ t` là bội chung** của `a` và `b`.
3. Dựng đồ thị phụ: nối mỗi `a ≤ t` với mọi bội `a, 2a, 3a, … ≤ t` (các "đỉnh giá trị"); hai số liên thông ⇔ liên thông trong đồ thị gốc.
4. Dùng DSU với nén đường + hợp theo kích thước; tổng số phép hợp `Σ t/a ≤ t · H_t`.
5. Đếm thành phần **chỉ trên các nút của `nums`**.

## 2. Đề bài & ví dụ

`n` nút, nút `i` mang giá trị `nums[i]` (đôi một khác nhau). Nút `i`, `j` có cạnh nếu `lcm(nums[i], nums[j]) ≤ threshold`. Đếm số thành phần liên thông.

**Ví dụ 1:** `nums = [2,4,8,3,9], threshold = 5` → `4` (`{2,4}`, `{3}`, `{8}`, `{9}`).

**Ví dụ 2:** `nums = [2,4,8,3,9,12], threshold = 10` → `2` (`{2,3,4,8,9}`, `{12}`).

**Ràng buộc:** `n ≤ 10^5`, `nums[i] ≤ 10^9`, `threshold ≤ 2·10^5`.

In [1]:
from typing import List
from math import gcd

examples = [([2, 4, 8, 3, 9], 5, 4), ([2, 4, 8, 3, 9, 12], 10, 2)]
for a, t, exp in examples:
    print(a, t, '-> kỳ vọng', exp)

[2, 4, 8, 3, 9] 5 -> kỳ vọng 4
[2, 4, 8, 3, 9, 12] 10 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. BCNN và bội chung

`lcm(a, b) = a·b / gcd(a, b)`; mọi bội chung của `a` và `b` là bội của `lcm(a, b)`. Vậy `lcm(a, b) ≤ t` ⇔ có bội chung `m ≤ t`.

In [2]:
def lcm(a, b):
    return a // gcd(a, b) * b

print(lcm(2, 3), lcm(4, 6), lcm(8, 3))

6 12 24


### 3.2. DSU (Union–Find)

In [3]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n)); self.sz = [1] * n

    def find(self, x):
        root = x
        while self.p[root] != root:
            root = self.p[root]
        while self.p[x] != root:                 # nén đường
            self.p[x], x = root, self.p[x]
        return root

    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b:
            return False
        if self.sz[a] < self.sz[b]:
            a, b = b, a
        self.p[b] = a; self.sz[a] += self.sz[b]
        return True

d = DSU(5); d.union(0, 1); d.union(3, 4); d.union(1, 4)
print([d.find(i) for i in range(5)])

[0, 0, 2, 0, 0]


### 3.3. Tổng điều hòa

Duyệt bội của mọi `a` trong `[1, t]`: `Σ_{a=1}^{t} t/a ≈ t ln t` ≈ `2.4·10^6` với `t = 2·10^5`.

In [4]:
t = 2 * 10 ** 5
print('Σ t // a =', sum(t // a for a in range(1, t + 1)))

Σ t // a = 2472113


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2 (`t = 10`): bội ≤ 10 — `2 → {2,4,6,8,10}`, `4 → {4,8}`, `8 → {8}`, `3 → {3,6,9}`, `9 → {9}`. Đỉnh giá trị `6` chung cho `2` và `3`; `8` chung cho `2, 4, 8`; `9` chung cho `3, 9` ⇒ `{2,3,4,8,9}` một thành phần. `12 > 10` cô lập ⇒ 2 thành phần.

In [5]:
for a in [2, 4, 8, 3, 9]:
    print(a, '->', list(range(a, 11, a)))

2 -> [2, 4, 6, 8, 10]
4 -> [4, 8]
8 -> [8]
3 -> [3, 6, 9]
9 -> [9]


## 5. Cách 1 — Brute Force: xét mọi cặp

In [6]:
def count_brute(nums, t):
    n = len(nums); d = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if lcm(nums[i], nums[j]) <= t:
                d.union(i, j)
    return len({d.find(i) for i in range(n)})

for a, t, exp in examples:
    print(count_brute(a, t), '| kỳ vọng', exp)

4 | kỳ vọng 4
2 | kỳ vọng 2


**Độ phức tạp:** `O(n² log)` — `10^{10}` với `n = 10^5`.

## 6. Ý tưởng thuật toán / bước đột phá

DSU trên các giá trị `1..t`. Với mỗi `a ∈ nums`, `a ≤ t`: hợp `a` với `2a, 3a, … ≤ t`. Đáp án = (số gốc khác nhau trong `{find(a) : a ∈ nums, a ≤ t}`) + (số phần tử `> t`).

**Vì sao đúng:** `a`, `b` có cạnh ⇒ chung đỉnh `lcm(a,b) ≤ t` ⇒ cùng thành phần. Ngược lại, đường đi trong đồ thị phụ xen kẽ số trong `nums` và đỉnh giá trị; mỗi đỉnh giá trị `m` nối các số chia hết `m` ⇒ hai số liền kề trên đường có bội chung `m ≤ t` ⇒ có cạnh trong đồ thị gốc.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS trên đồ thị "số ↔ bội"

In [7]:
from collections import deque

def count_bfs(nums, t):
    present = {a for a in nums if a <= t}
    big = sum(1 for a in nums if a > t)
    divs_present = [[] for _ in range(t + 1)]       # m -> các a ∈ nums chia hết m
    for a in present:
        for m in range(a, t + 1, a):
            divs_present[m].append(a)
    seen_num, seen_m = set(), [False] * (t + 1)
    comps = 0
    for a in present:
        if a in seen_num:
            continue
        comps += 1
        seen_num.add(a); q = deque([('n', a)])
        while q:
            kind, v = q.popleft()
            if kind == 'n':
                for m in range(v, t + 1, v):
                    if not seen_m[m]:
                        seen_m[m] = True; q.append(('m', m))
            else:
                for b in divs_present[v]:
                    if b not in seen_num:
                        seen_num.add(b); q.append(('n', b))
    return comps + big

for a, t, exp in examples:
    print(count_bfs(a, t), '| kỳ vọng', exp)

4 | kỳ vọng 4
2 | kỳ vọng 2


### 7.2. Cách 3 — DSU + sàng bội (lời giải chính)

In [8]:
def count_dsu(nums, t):
    d = DSU(t + 1)
    big = 0
    for a in nums:
        if a > t:
            big += 1
            continue
        for m in range(2 * a, t + 1, a):
            d.union(a, m)
    return len({d.find(a) for a in nums if a <= t}) + big

for a, t, exp in examples:
    print(count_dsu(a, t), '| kỳ vọng', exp)

4 | kỳ vọng 4
2 | kỳ vọng 2


### 7.3. Cách 4 — "Chủ sở hữu" của mỗi bội: chỉ hợp các số trong `nums`

Với mỗi `m ≤ t` ghi `owner[m]` = số đầu tiên trong `nums` chia hết `m`. Khi số `a` sau đó đi qua `m` đã có chủ, hợp `a` với `owner[m]`. DSU chỉ cần trên `n` nút (chỉ số trong `nums`).

In [9]:
def count_owner(nums, t):
    n = len(nums)
    d = DSU(n)
    owner = [-1] * (t + 1)
    big = 0
    for i, a in enumerate(nums):
        if a > t:
            big += 1
            continue
        for m in range(a, t + 1, a):
            if owner[m] == -1:
                owner[m] = i
            else:
                d.union(i, owner[m])
    return len({d.find(i) for i, a in enumerate(nums) if a <= t}) + big

for a, t, exp in examples:
    print(count_owner(a, t), '| kỳ vọng', exp)

4 | kỳ vọng 4
2 | kỳ vọng 2


## 8. TRACE từng bước

Trace cách 4 trên ví dụ 2.

In [10]:
def trace(nums, t):
    n = len(nums); d = DSU(n); owner = [-1] * (t + 1)
    for i, a in enumerate(nums):
        if a > t:
            print(f"{a} > {t}: cô lập"); continue
        events = []
        for m in range(a, t + 1, a):
            if owner[m] == -1:
                owner[m] = i
            else:
                if d.union(i, owner[m]):
                    events.append(f"bội {m} đã thuộc {nums[owner[m]]} -> hợp {a} với {nums[owner[m]]}")
        print(f"xử lý {a}: " + ('; '.join(events) if events else 'chưa hợp với ai'))
    groups = {}
    for i, a in enumerate(nums):
        groups.setdefault(d.find(i), []).append(a)
    print('thành phần:', list(groups.values()))

trace([2, 4, 8, 3, 9, 12], 10)

xử lý 2: chưa hợp với ai
xử lý 4: bội 4 đã thuộc 2 -> hợp 4 với 2
xử lý 8: bội 8 đã thuộc 2 -> hợp 8 với 2
xử lý 3: bội 6 đã thuộc 2 -> hợp 3 với 2
xử lý 9: bội 9 đã thuộc 3 -> hợp 9 với 3
12 > 10: cô lập
thành phần: [[2, 4, 8, 3, 9], [12]]


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def countComponents(self, nums: List[int], threshold: int) -> int:
        t = threshold
        parent = list(range(t + 1))

        def find(x):
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:
                parent[x], x = root, parent[x]
            return root

        big = 0
        for a in nums:
            if a > t:
                big += 1
                continue
            ra = find(a)
            for m in range(2 * a, t + 1, a):
                rm = find(m)
                if rm != ra:
                    parent[rm] = ra
        return len({find(a) for a in nums if a <= t}) + big

for a, t, exp in examples:
    got = Solution().countComponents(a, t)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
2 OK


Trong bản nộp, gốc `ra` của `a` không đổi trong vòng lặp vì ta luôn treo gốc khác **vào** `ra`.

## 10. Vì sao thuật toán đúng

1. **Số lớn:** `a > t` ⇒ `lcm(a, b) ≥ a > t` ⇒ không có cạnh.
2. **Tương đương liên thông:** như mục 6 — đồ thị phụ (số ↔ bội ≤ t) có cùng các thành phần trên tập số `≤ t` với đồ thị gốc.
3. **Không đếm đỉnh giá trị:** chỉ đếm gốc của các phần tử của `nums`; đỉnh giá trị chỉ làm "cầu nối".
4. Giá trị đôi một khác nhau nên mỗi đỉnh giá trị `a` đại diện đúng một nút.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp | `O(n² log V)` | `O(n)` |
| 2. BFS số ↔ bội | `O(t log t)` | `O(t log t)` |
| 3. DSU + sàng bội | `O(t log t · α)` | `O(t)` |
| 4. DSU trên `nums` + chủ sở hữu | `O(t log t · α)` | `O(n + t)` |

Ghi chú: tổng số bước của vòng bội là `Σ_{a ∈ nums, a ≤ t} t/a`. Vì các giá trị đôi một khác nhau, tổng này không vượt `Σ_{a=1}^{t} t/a = t · H_t ≈ 2,5·10^6` — kể cả trường hợp xấu nhất có số `1` (riêng nó đã `t` bước, và nó nối mọi số `≤ t` thành một thành phần). Phần còn lại (`find`) gần như hằng số nhờ nén đường.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'bfs': count_bfs, 'dsu': count_dsu, 'owner': count_owner,
         'submit': lambda a, t: Solution().countComponents(a, t)}
edge = [([1], 1, 1), ([5], 3, 1), ([1, 7, 9], 10, 1), ([2, 3], 5, 2), ([2, 3], 6, 1), ([4, 6], 12, 1), ([4, 6], 11, 2)]
for a, t, exp in examples + edge:
    assert count_brute(a, t) == exp, (a, t, count_brute(a, t))
    for name, f in impls.items():
        assert f(a, t) == exp, (name, a, t)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [13]:
import random
rng = random.Random(3378)
cnt = 0
for _ in range(600):
    t = rng.randint(1, 60)
    nums = rng.sample(range(1, 90), rng.randint(1, 15))
    exp = count_brute(nums, t)
    for name, f in impls.items():
        assert f(nums, t) == exp, (name, nums, t)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

Khớp vét cạn trên 600 bộ ngẫu nhiên.


### 12.3. Ca lớn

In [14]:
import time
t = 2 * 10 ** 5
cases = {
    'mọi số 1..10^5, t = 2e5': list(range(1, 10 ** 5 + 1)),
    'ngẫu nhiên tới 1e9': rng.sample(range(1, 10 ** 9), 10 ** 5),
    'ngẫu nhiên tới 4e5': rng.sample(range(1, 4 * 10 ** 5), 10 ** 5),
}
for name, nums in cases.items():
    t0 = time.time(); a = Solution().countComponents(nums, t); t1 = time.time() - t0
    t0 = time.time(); b = count_owner(nums, t); t2 = time.time() - t0
    assert a == b
    print(f"{name:26s}: {a:6d} | DSU giá trị {t1:.2f}s | chủ sở hữu {t2:.2f}s")

mọi số 1..10^5, t = 2e5   :      1 | DSU giá trị 0.36s | chủ sở hữu 0.65s
ngẫu nhiên tới 1e9        : 100000 | DSU giá trị 0.01s | chủ sở hữu 0.01s


ngẫu nhiên tới 4e5        :  49886 | DSU giá trị 0.22s | chủ sở hữu 0.27s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ nối hai số khi một số chia hết số kia

In [15]:
def wrong_divisibility(nums, t):
    n = len(nums); d = DSU(n)
    for i in range(n):
        for j in range(n):
            if i != j and nums[j] % nums[i] == 0 and nums[j] <= t:
                d.union(i, j)
    return len({d.find(i) for i in range(n)})

print('sai :', wrong_divisibility([2, 3], 6), '| đúng:', Solution().countComponents([2, 3], 6))

sai : 2 | đúng: 1


**Vì sao sai:** `lcm(2, 3) = 6 ≤ 6` dù không số nào chia hết số kia. Phải nối qua **bội chung**.

### Lỗi 2 — Quên đếm các số `> threshold`

In [16]:
def wrong_drop_big(nums, t):
    d = DSU(t + 1)
    for a in nums:
        if a <= t:
            for m in range(2 * a, t + 1, a):
                d.union(a, m)
    return len({d.find(a) for a in nums if a <= t})

print('sai :', wrong_drop_big([2, 4, 8, 3, 9, 12], 10), '| đúng:', Solution().countComponents([2, 4, 8, 3, 9, 12], 10))

sai : 1 | đúng: 2


**Vì sao sai:** mỗi số lớn là một thành phần riêng.

### Lỗi 3 — Đếm thành phần trên **mọi** đỉnh giá trị `1..t`

In [17]:
def wrong_count_all_values(nums, t):
    d = DSU(t + 1)
    big = 0
    for a in nums:
        if a > t:
            big += 1; continue
        for m in range(2 * a, t + 1, a):
            d.union(a, m)
    return len({d.find(v) for v in range(1, t + 1)}) + big

print('sai :', wrong_count_all_values([2, 4, 8, 3, 9], 5), '| đúng:', Solution().countComponents([2, 4, 8, 3, 9], 5))

sai : 6 | đúng: 4


**Vì sao sai:** giá trị không có trong `nums` (như 1, 5) chỉ là đỉnh phụ, không phải nút của đồ thị.

### Lỗi 4 — Dùng tích `a · b ≤ t` thay cho BCNN

In [18]:
def wrong_product(nums, t):
    n = len(nums); d = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] * nums[j] <= t:
                d.union(i, j)
    return len({d.find(i) for i in range(n)})

print('sai :', wrong_product([4, 6], 12), '| đúng:', Solution().countComponents([4, 6], 12))

sai : 2 | đúng: 1


**Vì sao sai:** `lcm(4, 6) = 12` nhưng `4 · 6 = 24`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện kiểu `lcm ≤ t` / `gcd > 1` giữa các cặp ⇒ nối qua **bội/ước** chung bằng đỉnh phụ, không duyệt cặp.
- Sàng bội có tổng điều hòa `O(t log t)`.
- Giá trị lớn hơn ngưỡng ⇒ xử lý riêng (cô lập).
- DSU trên giá trị khi giá trị nhỏ; trên chỉ số + "chủ sở hữu" khi muốn tiết kiệm.

## 15. Biến thể của bài

1. **LC 952 — Largest Component Size by Common Factor:** cạnh khi `gcd > 1` ⇒ nối qua **ước nguyên tố**.
2. **LC 1627 — Graph Connectivity With Threshold:** cạnh khi có ước chung `> threshold`.
3. Đếm kích thước thành phần lớn nhất thay vì số thành phần.
4. Cạnh khi `gcd(a, b) ≥ k` ⇒ nối mỗi `d ≥ k` với các bội của nó có trong mảng.

In [19]:
# Biến thể 2 (LC 1627): city a, b nối nếu có ước chung > threshold; trả lời truy vấn
def are_connected(n, threshold, queries):
    d = DSU(n + 1)
    for z in range(threshold + 1, n + 1):
        for m in range(2 * z, n + 1, z):
            d.union(z, m)
    return [d.find(a) == d.find(b) for a, b in queries]

assert are_connected(6, 2, [[1, 4], [2, 5], [3, 6]]) == [False, False, True]
assert are_connected(6, 0, [[4, 5], [3, 4], [3, 2], [2, 6], [1, 3]]) == [True] * 5
print('LC 1627 ví dụ: đúng.')

LC 1627 ví dụ: đúng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi cặp | `O(n²)` | Oracle |
| 2 | BFS số ↔ bội | `O(t log t)` | Không cần DSU |
| 3 | DSU trên giá trị | `O(t log t)` | **Nộp bài** |
| 4 | DSU trên chỉ số + chủ sở hữu | `O(t log t)` | Tiết kiệm nút |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1, 2, 3, 4]`, `threshold = 4`.

**Bài 2 (tính tay):** `nums = [6, 10, 15]`, `threshold = 30`.

**Bài 3 (điền TODO):** trả về **kích thước** thành phần lớn nhất.

**Bài 4 (tự viết lại):** LC 952 — nối khi `gcd > 1`.

**Bài 5 (mini challenge):** nhiều truy vấn `threshold` khác nhau (tăng dần) trên cùng `nums` — trả lời số thành phần cho mỗi truy vấn.

In [20]:
def largest_component_todo(nums, t):
    # TODO
    raise NotImplementedError

def largest_by_common_factor_todo(nums):
    # TODO
    raise NotImplementedError

def components_for_thresholds_todo(nums, thresholds):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `1` nối với mọi số ≤ 4 (lcm = số đó). (2) Một thành phần. (3) `1`.

**Bài 2.** (1) `lcm(6, 10) = 30`, `lcm(6, 15) = 30`, `lcm(10, 15) = 30`. (2) Đều ≤ 30. (3) `1`.

**Bài 3.** (1) Đếm số phần tử `nums` theo gốc. (2) Lấy max. (3) Số lớn là thành phần cỡ 1.

**Bài 4.** (1) Phân tích thừa số nguyên tố mỗi số. (2) Hợp số với các ước nguyên tố. (3) Đếm kích thước.

**Bài 5.** (1) Ngưỡng tăng ⇒ cạnh chỉ thêm, không mất. (2) Xử lý các đỉnh giá trị `m` tăng dần, thêm cạnh khi `m ≤ t` hiện tại. (3) Theo dõi số thành phần bằng DSU (mỗi hợp thành công giảm 1).

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
assert Solution().countComponents([1, 2, 3, 4], 4) == 1 and Solution().countComponents([6, 10, 15], 30) == 1

def largest_component(nums, t):
    n = len(nums); d = DSU(n); owner = [-1] * (t + 1)
    for i, a in enumerate(nums):
        if a > t:
            continue
        for m in range(a, t + 1, a):
            if owner[m] == -1: owner[m] = i
            else: d.union(i, owner[m])
    sizes = {}
    for i in range(n):
        r = d.find(i); sizes[r] = sizes.get(r, 0) + 1
    return max(sizes.values())

assert largest_component([2, 4, 8, 3, 9, 12], 10) == 5
print('Bài 1: 1 | Bài 2: 1 | Bài 3:', largest_component([2, 4, 8, 3, 9, 12], 10))

Bài 1: 1 | Bài 2: 1 | Bài 3: 5


In [22]:
# Bài 4, 5
def largest_by_common_factor(nums):
    M = max(nums)
    spf = list(range(M + 1))
    for i in range(2, int(M ** 0.5) + 1):
        if spf[i] == i:
            for j in range(i * i, M + 1, i):
                if spf[j] == j:
                    spf[j] = i
    d = DSU(M + 1)
    for a in nums:
        x = a
        while x > 1:
            p = spf[x]; d.union(a, p)
            while x % p == 0:
                x //= p
    cnt = {}
    for a in nums:
        r = d.find(a); cnt[r] = cnt.get(r, 0) + 1
    return max(cnt.values())

assert largest_by_common_factor([4, 6, 15, 35]) == 4 and largest_by_common_factor([20, 50, 9, 63]) == 2
assert largest_by_common_factor([2, 3, 6, 7, 4, 12, 21, 39]) == 8                    # ví dụ LC 952

def components_for_thresholds(nums, thresholds):
    T = max(thresholds)
    idx_of = {a: i for i, a in enumerate(nums)}
    n = len(nums); d = DSU(n)
    owner = {}                                      # đỉnh m -> chỉ số đầu tiên chia hết m
    comps = n
    events = []                                     # (m, i): số nums[i] chia hết m, có hiệu lực khi m <= t
    for i, a in enumerate(nums):
        if a <= T:
            for m in range(a, T + 1, a):
                events.append((m, i))
    events.sort()
    out = {}; ptr = 0
    for t in sorted(set(thresholds)):
        while ptr < len(events) and events[ptr][0] <= t:
            m, i = events[ptr]; ptr += 1
            if m in owner:
                if d.union(i, owner[m]):
                    comps -= 1
            else:
                owner[m] = i
        out[t] = comps
    return [out[t] for t in thresholds]

rng = random.Random(5)
nums = rng.sample(range(1, 60), 20)
ths = [3, 10, 25, 40, 60]
assert components_for_thresholds(nums, ths) == [count_brute(nums, t) for t in ths]
print('Bài 4: LC 952 ví dụ đúng | Bài 5 khớp vét cạn:', components_for_thresholds(nums, ths))

Bài 4: LC 952 ví dụ đúng | Bài 5 khớp vét cạn: [20, 18, 15, 11, 5]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
parent = list(range(t + 1)); big = 0
for a in nums:
    if a > t: big += 1; continue
    for m in range(2a, t + 1, a): union(a, m)
return len({find(a) for a in nums if a <= t}) + big
```

**Checklist:**

- [ ] Số `> threshold` đếm riêng (mỗi số một thành phần)?
- [ ] Nối qua **bội chung** (không chỉ quan hệ chia hết)?
- [ ] Chỉ đếm gốc của các phần tử `nums`?
- [ ] Dùng BCNN, không dùng tích?
- [ ] DSU có nén đường (tránh chậm)?